# 142 — Semantic Caching

**What you'll learn:**
- How semantic caching differs from exact-match caching — and why it matters in production
- Cosine similarity: the math and intuition for measuring how close two embedding vectors are
- Threshold sensitivity: what breaks at 0.85 / 0.92 / 0.99 and how to choose the right value
- Cache eviction strategies: LRU, time-based (TTL), and capacity-based approaches
- Embedding model tradeoffs: speed vs cost vs quality across OpenAI and local options
- How to wire a `SemanticCache` into a LangGraph workflow

---

**Two sections:**
- **Part A** — CPU-safe concept demonstrations. No API key required. Uses hand-crafted vectors.
- **Part B** — Live OpenAI API calls. Requires `OPENAI_API_KEY`.

---

`Source: examples/142-semantic-caching/ | Model: text-embedding-3-small, gpt-5.4-nano | Key: OPENAI_API_KEY`

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A — CPU-Safe Concept Demonstrations

Everything in Part A runs with **no API key**. We use hand-crafted vectors to make the mechanics
transparent. Real embeddings behave identically — they're just 1536-dimensional instead of 4-dimensional.

### Concept — Semantic vs Exact-Match Caching

**Exact-match caching** stores a response keyed by the exact query string (or its hash).
If the user rephrases — even slightly — there is no hit.

**Semantic caching** embeds each query into a vector and checks whether any cached vector
is *close enough* in meaning. Paraphrases, synonyms, and minor rewording all hit the same entry.

| Property | Exact-match cache | Semantic cache |
|---|---|---|
| Key type | String hash | Embedding vector |
| Hit condition | Identical query | Similarity ≥ threshold |
| Storage | `Dict[str, str]` | `List[(vector, response)]` |
| Lookup cost | O(1) hash lookup | O(N) dot products |
| Hit rate | Low (wording-sensitive) | High (meaning-sensitive) |
| Risk | Stale exact responses | Slightly imprecise responses |

Semantic caching trades a small risk of false positives for dramatically higher hit rates
in real applications where users rarely type the exact same query twice.

### Concept — Cosine Similarity

Cosine similarity measures the **angle** between two vectors, not the distance between them.
This makes it magnitude-invariant — useful for text embeddings, which vary in length.

**Formula:**
```
cos(θ) = (A · B) / (‖A‖ × ‖B‖)
```

**ASCII diagram — two vectors and the angle between them:**
```
     B
    /
   /  θ  (small angle = high similarity)
  /
 A————————→
```

**Range:**
- `1.0` — vectors point in the same direction (identical meaning)
- `0.0` — vectors are orthogonal (completely unrelated topics)
- `-1.0` — vectors point in opposite directions (rare for text; embeddings are usually positive)

**Why cosine over Euclidean distance?**
Two sentences that express the same idea but use different numbers of tokens will produce
embedding vectors of different magnitudes. Cosine ignores magnitude — it only cares about
direction, which captures semantic meaning.

### Concept — Threshold Sensitivity

The threshold is the minimum similarity score required to call a lookup a **cache hit**.
Choosing it wrong causes either too many misses (wasted LLM calls) or false positives
(serving a subtly wrong answer).

| Threshold | Behavior | Risk |
|---|---|---|
| 0.85 | Many hits, very permissive | May return imprecise answers for loosely related queries |
| 0.92 (default) | Balanced hit rate and precision | Good tradeoff for factual Q&A |
| 0.99 | Very strict, few hits | Only near-identical queries hit; minimal false positives |

> **Rule of thumb:** For factual Q&A, 0.92 is safe. For creative or nuanced tasks
> where phrasing changes meaning (poetry, legal, medical), use 0.96+.

### Concept — Cache Eviction Strategies

Without eviction, a semantic cache grows unbounded. Four strategies:

| Strategy | How it works | Best for |
|---|---|---|
| **LRU** (Least Recently Used) | Evict the entry not accessed for the longest time | General purpose — topics shift over time |
| **TTL** (Time-To-Live) | Evict entries after N seconds regardless of access | Frequently changing data (news, prices) |
| **Capacity-based** | Evict oldest entry when cache exceeds N entries | Memory-constrained systems |
| **Score-based** | Evict lowest-similarity entries first | Quality-focused caches where precision matters |

Combining **TTL** (freshness) + **LRU** (relevance) is the most common production approach.
This example implements capacity-based and TTL variants in the exercises.

### Concept — Embedding Model Choices

The embedding model determines the quality of the similarity signal. Larger dimensions
generally capture more nuance, but cost more and are slower to embed and compare.

| Model | Dims | Cost/MTok | Speed | Quality |
|---|---|---|---|---|
| `text-embedding-3-small` | 1536 | $0.02 | Fast | Good — best value |
| `text-embedding-3-large` | 3072 | $0.13 | Slower | Better for high-precision search |
| `text-embedding-ada-002` | 1536 | $0.10 | Fast | Dated — prefer 3-small |
| `all-MiniLM-L6-v2` (local) | 384 | Free | Very fast | Good for short text, no API call |

> For a semantic cache, **`text-embedding-3-small`** is the sweet spot: cheap, fast, and
> accurate enough that the embedding cost is negligible compared to the LLM calls it avoids.
> At $0.02/MTok, embedding 1000 queries of ~50 tokens each costs $0.001 total.

In [ ]:
# Part A — cosine_sim() function demo
# This is the exact cosine_sim() from src/tools.py.

import math

def cosine_sim(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b + 1e-10)


# Identical vectors → similarity = 1.0
v1 = [0.8, 0.1, 0.5, 0.3]
v2 = [0.8, 0.1, 0.5, 0.3]
sim = cosine_sim(v1, v2)
print(f"Identical vectors:        {sim:.4f}  → identical meaning")

# Similar vectors (small perturbation) → high similarity
v3 = [0.81, 0.09, 0.51, 0.29]
sim = cosine_sim(v1, v3)
print(f"Similar (minor tweak):    {sim:.4f}  → {'HIT at 0.92 threshold' if sim >= 0.92 else 'MISS at 0.92'}")

# Different topic vectors → low similarity
v4 = [0.1, 0.9, 0.0, 0.8]
sim = cosine_sim(v1, v4)
print(f"Different topic:          {sim:.4f}  → {'HIT at 0.92 threshold' if sim >= 0.92 else 'MISS at 0.92'}")

# Orthogonal vectors (perpendicular) → similarity = 0.0
v5 = [0.0, 0.0, 1.0, 0.0]
v6 = [1.0, 0.0, 0.0, 0.0]
sim = cosine_sim(v5, v6)
print(f"Orthogonal vectors:       {sim:.4f}  → completely unrelated (angle = 90°)")

# Slightly noisy same-direction vector → shows robustness
import random; random.seed(42)
v7 = [x + random.uniform(-0.02, 0.02) for x in v1]
sim = cosine_sim(v1, v7)
print(f"±2% noise on v1:           {sim:.4f}  → {'HIT' if sim >= 0.92 else 'MISS'} at 0.92 (real paraphrase range)")

print("\nInterpretation: 0.0 = unrelated | 0.92+ = same meaning | 1.0 = identical")

In [ ]:
# Part A — SemanticCache class walkthrough
# Same SemanticCache from src/tools.py with internal state printed after each insertion.

from dataclasses import dataclass, field

@dataclass
class SemanticCache:
    threshold: float = 0.92
    _entries: list = field(default_factory=list)
    hits: int = 0
    misses: int = 0

    def get(self, query_vec: list[float]) -> str | None:
        best = max(((cosine_sim(query_vec, vec), response) for vec, response in self._entries), default=(0.0, None), key=lambda match: match[0])
        if best[0] >= self.threshold:
            self.hits += 1
            return best[1]
        self.misses += 1
        return None

    def set(self, query_vec: list[float], response: str) -> None:
        self._entries.append((query_vec, response))

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": round(self.hits / total, 3) if total else 0,
        }


# Show internal state as entries accumulate
cache = SemanticCache(threshold=0.92)

entries_to_add = [
    ([0.9, 0.1, 0.0, 0.1], "ML is a field of AI that learns from data."),
    ([0.7, 0.6, 0.1, 0.0], "Deep learning uses many-layered neural networks."),
    ([0.0, 0.0, 0.9, 0.8], "A relational database stores data in tables."),
]

print("Inserting entries and showing cache state:\n")
for i, (vec, resp) in enumerate(entries_to_add, 1):
    cache.set(vec, resp)
    print(f"After insertion {i}: cache has {len(cache._entries)} entries")
    for j, (v, r) in enumerate(cache._entries):
        print(f"  [{j}] vec[:2]={[round(x,2) for x in v[:2]]}  response='{r[:40]}'")
    print()

print("Lookup mechanics:")
test_vec = [0.89, 0.11, 0.01, 0.09]  # near-duplicate of ML entry
result = cache.get(test_vec)
print(f"  Query vec (ML paraphrase): result = '{result[:50] if result else None}'")
print(f"  Stats: {cache.stats()}")

In [ ]:
# Part A — Threshold sweep with mock queries
# 6 queries across 3 semantic clusters, each cluster has a near-duplicate.
# Shows how threshold changes hit/miss behavior.

def run_threshold_demo(threshold: float, query_set: list[tuple]) -> list[tuple]:
    """Return list of (query_text, decision) for a given threshold."""
    cache = SemanticCache(threshold=threshold)
    results = []
    for query_text, vec in query_set:
        cached = cache.get(vec)
        if cached:
            results.append((query_text, "HIT"))
        else:
            cache.set(vec, f"answer:{query_text[:20]}")
            results.append((query_text, "MISS"))
    return results


# 3 clusters; near-duplicate similarity ~0.9997 within cluster, ~0.3 across clusters
QUERY_SET = [
    ("What is machine learning?",        [0.900, 0.100, 0.000, 0.100]),
    ("Explain machine learning",         [0.895, 0.102, 0.002, 0.098]),  # sim ~0.9997
    ("What is deep learning?",           [0.700, 0.600, 0.100, 0.000]),
    ("How does deep learning work?",     [0.698, 0.601, 0.099, 0.001]),  # sim ~0.9999
    ("What is a relational database?",   [0.000, 0.050, 0.900, 0.800]),
    ("Explain relational databases",     [0.001, 0.048, 0.901, 0.799]),  # sim ~0.9999
]

print(f"{'Query':<40}  {'0.85':>6}  {'0.92':>6}  {'0.99':>6}")
print("-" * 64)

results_by_threshold = {
    t: run_threshold_demo(t, QUERY_SET) for t in [0.85, 0.92, 0.99]
}

for i, (query_text, _) in enumerate(QUERY_SET):
    r85 = results_by_threshold[0.85][i][1]
    r92 = results_by_threshold[0.92][i][1]
    r99 = results_by_threshold[0.99][i][1]
    print(f"  {query_text:<38}  {r85:>6}  {r92:>6}  {r99:>6}")

print()
for t in [0.85, 0.92, 0.99]:
    hits = sum(1 for _, d in results_by_threshold[t] if d == "HIT")
    print(f"Threshold {t}: {hits}/6 hits  ({hits/6:.0%} hit rate)")

In [ ]:
# Part A — Trace table: 5 queries, 2 near-duplicates
# Trace each decision: query | similarity to best match | decision | response snippet

THRESHOLD = 0.92

LLM_ANSWERS = {
    "What is machine learning?": "ML enables computers to learn patterns from data without explicit programming.",
    "What is deep learning?": "Deep learning uses neural networks with many layers to learn representations.",
    "What is a relational database?": "A relational DB organizes data into tables linked by foreign keys.",
}

QUERIES = [
    ("What is machine learning?",         [0.900, 0.100, 0.000, 0.100]),  # Q1: new topic → MISS → store
    ("Can you explain machine learning?", [0.892, 0.108, 0.003, 0.097]),  # Q2: paraphrase of Q1 → HIT
    ("What is deep learning?",            [0.700, 0.600, 0.100, 0.000]),  # Q3: new topic → MISS → store
    ("How does deep learning work?",      [0.695, 0.605, 0.098, 0.002]),  # Q4: paraphrase of Q3 → HIT
    ("What is a relational database?",    [0.000, 0.050, 0.900, 0.800]),  # Q5: unrelated → MISS → store
]

cache = SemanticCache(threshold=THRESHOLD)

print(f"{'#':<2}  {'Query':<38}  {'Best Sim':>8}  {'Decision':<8}  Response snippet")
print("-" * 95)

for i, (query, vec) in enumerate(QUERIES, 1):
    best_sim = max(
        (cosine_sim(vec, cached_vec) for cached_vec, _ in cache._entries),
        default=0.0
    )
    result = cache.get(vec)
    if result:
        decision = "HIT"
        response = result
    else:
        decision = "MISS"
        response = LLM_ANSWERS.get(query, f"[LLM: {query[:30]}...]")
        cache.set(vec, response)
    print(f"{i:<2}  {query:<38}  {best_sim:>8.4f}  {decision:<8}  {response[:45]}")

print()
print(f"Final stats: {cache.stats()}")
print(f"LLM calls avoided: {cache.hits} of {len(QUERIES)} queries")

### Production Deployment Pattern

In production, a semantic cache sits in front of the LLM API. Every query is embedded first,
then checked against the cache before hitting the LLM.

```
User query
    │
    ▼
Embed query  ──►  vector similarity scan  ──► HIT?  ──► return cached answer
                                              │ MISS
                                              ▼
                                       LLM API call
                                              │
                                              ▼
                               store (embedding, answer) in cache
                                              │
                                              ▼
                                       return answer
```

**Production backends** (replace the in-memory list in this example):
- **Redis + RediSearch** — low-latency, persistent, horizontal scaling
- **Qdrant / Milvus** — dedicated vector DB, HNSW index for O(log n) lookup
- **pgvector** — Postgres extension, stays in your existing DB stack
- **In-memory (this example)** — fine for ≤ 10k entries, lost on restart

In [ ]:
# Part A — Capacity-based LRU eviction
# Shows how to bound cache size and evict the least-recently-used entry.
# collections.OrderedDict preserves insertion order and supports move_to_end().

from collections import OrderedDict

class SemanticCacheLRU:
    """Semantic cache with LRU eviction when max_size is reached."""

    def __init__(self, threshold: float = 0.92, max_size: int = 5):
        self.threshold = threshold
        self.max_size = max_size
        # OrderedDict: key = tuple(vec), value = response
        # Most recently used entry is at the END (move_to_end default)
        self._store: OrderedDict = OrderedDict()
        self.hits = 0
        self.misses = 0
        self.evictions = 0

    def get(self, query_vec: list[float]) -> str | None:
        for key_vec_tuple, response in list(self._store.items()):
            if cosine_sim(query_vec, list(key_vec_tuple)) >= self.threshold:
                self._store.move_to_end(key_vec_tuple)  # mark as recently used
                self.hits += 1
                return response
        self.misses += 1
        return None

    def set(self, query_vec: list[float], response: str) -> None:
        key = tuple(query_vec)
        if key in self._store:
            self._store.move_to_end(key)
        else:
            if len(self._store) >= self.max_size:
                self._store.popitem(last=False)  # remove LRU (first / oldest item)
                self.evictions += 1
        self._store[key] = response

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            "size": len(self._store),
            "hits": self.hits,
            "misses": self.misses,
            "evictions": self.evictions,
            "hit_rate": round(self.hits / total, 3) if total else 0,
        }


# Demo: max_size=3, add 5 entries to trigger 2 evictions
lru = SemanticCacheLRU(threshold=0.92, max_size=3)
vecs   = [[1,0,0,0], [0,1,0,0], [0,0,1,0], [0,0,0,1], [0.5,0.5,0,0]]
topics = ["ML", "DL", "Python", "Databases", "AI overview"]

for vec, topic in zip(vecs, topics):
    lru.set(vec, f"Answer about {topic}")
    print(f"After adding '{topic}': size={len(lru._store)}, evictions={lru.evictions}")
    print(f"  Cached topics: {[list(k)[:2] for k in lru._store.keys()]}")

print(f"\nFinal stats: {lru.stats()}")
print("LRU keeps the cache bounded while retaining recently active entries.")

In [ ]:
# Part A — Threshold precision analysis
# Shows how threshold choice creates false positives at low values.
# Each test pair has a known similarity and whether a hit would be semantically correct.

# (description, similarity_score, is_correct_hit)
# Similarity scores approximate what text-embedding-3-small returns for these query pairs.
TEST_PAIRS = [
    ("exact same query",                           1.00, True),
    ("near-identical wording",                     0.97, True),
    ("clear paraphrase, same intent",              0.94, True),
    ("same topic, slightly different intent",      0.91, True),   # borderline
    ("related topic, different specific question", 0.88, False),  # trap: sounds close
    ("same domain, different question",            0.84, False),  # false positive risk
    ("tangentially related query",                 0.75, False),
    ("completely different topic",                 0.40, False),
]

print(f"{'Threshold':>10} | {'Hits':>4} | {'Correct hits':>12} | {'False+':>6} | {'Precision':>9}")
print("-" * 52)

for thresh in [0.80, 0.85, 0.90, 0.92, 0.94, 0.96, 0.99]:
    matched = [(desc, correct) for desc, sim, correct in TEST_PAIRS if sim >= thresh]
    total_hits = len(matched)
    true_pos  = sum(1 for _, correct in matched if correct)
    false_pos = sum(1 for _, correct in matched if not correct)
    precision = true_pos / total_hits if total_hits else 1.0
    print(f"{thresh:>10.2f} | {total_hits:>4} | {true_pos:>12} | {false_pos:>6} | {precision:>9.0%}")

print()
print("Recommendation: 0.92 gives 100% precision while still catching clear paraphrases.")
print("Below 0.90, false positives appear — semantically related but distinct questions may get wrong answers.")

### Exercise 1 — Track Global Hit/Miss Counters

The current `SemanticCache` tracks `hits` and `misses` as **instance attributes**.
Each cache instance has its own counters — if you create two caches, you can't see the
combined hit rate across both.

**Exercise:** Add `hit_count` and `miss_count` as **class-level** counters so ALL instances
share a single global tally. Use a `@classmethod reset()` to clear them.

**Why this matters:** In production you might have a warm cache and a test cache running
simultaneously. Class-level counters let you measure the aggregate hit rate without
passing a reference between cache instances.

**Tradeoff:** Class-level counters are not thread-safe without a lock. For single-threaded
or async code (like this LangGraph workflow), they are fine.

In [ ]:
# Exercise 1 — TODO stubs
# Implement SemanticCacheGlobal with class-level counters.

class SemanticCacheGlobal:
    # TODO: add class-level hit_count and miss_count here
    # hit_count = 0
    # miss_count = 0

    def __init__(self, threshold: float = 0.92):
        self.threshold = threshold
        self._entries: list = []

    def get(self, query_vec: list[float]) -> str | None:
        for vec, response in self._entries:
            if cosine_sim(query_vec, vec) >= self.threshold:
                # TODO: increment SemanticCacheGlobal.hit_count
                return response
        # TODO: increment SemanticCacheGlobal.miss_count
        return None

    def set(self, query_vec: list[float], response: str) -> None:
        self._entries.append((query_vec, response))

    @classmethod
    def reset(cls) -> None:
        """Reset global counters."""
        # TODO: set hit_count and miss_count back to 0
        pass

    @classmethod
    def global_hit_rate(cls) -> float:
        # TODO: return hit_count / (hit_count + miss_count)
        return 0.0

print("Implement the TODOs above, then run the answer key cell below.")

In [ ]:
# Exercise 1 — Answer Key

class SemanticCacheGlobal:
    # Class-level counters: shared across ALL instances
    hit_count: int = 0
    miss_count: int = 0

    def __init__(self, threshold: float = 0.92):
        self.threshold = threshold
        self._entries: list = []

    def get(self, query_vec: list[float]) -> str | None:
        for vec, response in self._entries:
            if cosine_sim(query_vec, vec) >= self.threshold:
                SemanticCacheGlobal.hit_count += 1  # class-level increment
                return response
        SemanticCacheGlobal.miss_count += 1  # class-level increment
        return None

    def set(self, query_vec: list[float], response: str) -> None:
        self._entries.append((query_vec, response))

    @classmethod
    def reset(cls) -> None:
        cls.hit_count = 0
        cls.miss_count = 0

    @classmethod
    def global_hit_rate(cls) -> float:
        total = cls.hit_count + cls.miss_count
        return cls.hit_count / total if total else 0.0


# Prove two separate instances share the same counters
SemanticCacheGlobal.reset()
cache_a = SemanticCacheGlobal(threshold=0.92)
cache_b = SemanticCacheGlobal(threshold=0.92)

vec_ml = [1.0, 0.0, 0.0]
cache_a.set(vec_ml, "ML answer from cache_a")

cache_a.get([0.99, 0.1, 0.0])   # HIT in cache_a
cache_b.get([0.5, 0.5, 0.5])    # MISS in cache_b
cache_a.get([1.0, 0.0, 0.0])    # HIT in cache_a

print(f"cache_a sees: hit_count={SemanticCacheGlobal.hit_count}, miss_count={SemanticCacheGlobal.miss_count}")
print(f"cache_b sees: hit_count={SemanticCacheGlobal.hit_count}, miss_count={SemanticCacheGlobal.miss_count}")
print(f"Global hit rate: {SemanticCacheGlobal.global_hit_rate():.0%}")
print()
# Tradeoff: class-level counters are easy to aggregate but NOT thread-safe without a Lock.
# For concurrent workloads: use threading.Lock() or an atomic counter from a library.
print("Note: class-level counters are convenient but not thread-safe under concurrent writes.")

### Exercise 2 — Add TTL Expiry

The current `SemanticCache` has **no time-based expiry**. A response cached at startup
will be served indefinitely — even if the underlying facts change.

**Exercise:** Add a `ttl_seconds` parameter so entries expire after N seconds.
The `get()` method should skip expired entries.

**Implementation hint:** Store `time.time()` alongside each entry at insertion.
In `get()`, check `time.time() - insertion_time > ttl_seconds` before comparing similarity.

**Thread-safety note:** `time.time()` reads are atomic in CPython. Writing to `self._entries`
from multiple threads simultaneously would require a `threading.Lock()`. For async LangGraph
workflows this is not a concern.

In [ ]:
# Exercise 2 — TODO stubs

import time

class SemanticCacheTTL:
    def __init__(self, threshold: float = 0.92, ttl_seconds: float = 300.0):
        self.threshold = threshold
        self.ttl_seconds = ttl_seconds
        self._entries: list = []  # will store (vec, response, timestamp)

    def set(self, query_vec: list[float], response: str) -> None:
        # TODO: append (query_vec, response, time.time()) to _entries
        pass

    def get(self, query_vec: list[float]) -> str | None:
        now = time.time()
        for entry in self._entries:
            # TODO: unpack (vec, response, timestamp)
            # TODO: skip if now - timestamp > self.ttl_seconds
            # TODO: return response if cosine_sim >= threshold
            pass
        return None

print("Implement the TODOs above, then run the answer key cell below.")

In [ ]:
# Exercise 2 — Answer Key

import time

class SemanticCacheTTL:
    def __init__(self, threshold: float = 0.92, ttl_seconds: float = 300.0):
        self.threshold = threshold
        self.ttl_seconds = ttl_seconds
        self._entries: list = []  # stores (vec, response, inserted_at)

    def set(self, query_vec: list[float], response: str) -> None:
        self._entries.append((query_vec, response, time.time()))  # record insertion time

    def get(self, query_vec: list[float]) -> str | None:
        now = time.time()
        for vec, response, inserted_at in self._entries:
            if now - inserted_at > self.ttl_seconds:
                continue  # expired — skip silently
            if cosine_sim(query_vec, vec) >= self.threshold:
                return response
        return None

    def evict_expired(self) -> int:
        """Explicitly remove expired entries. Returns count evicted."""
        now = time.time()
        before = len(self._entries)
        self._entries = [e for e in self._entries if now - e[2] <= self.ttl_seconds]
        return before - len(self._entries)


# Demo: very short TTL to show expiry
cache_ttl = SemanticCacheTTL(threshold=0.90, ttl_seconds=0.05)  # 50ms TTL
cache_ttl.set([1.0, 0.0, 0.0], "Fresh answer about ML")

result = cache_ttl.get([1.0, 0.0, 0.0])
print(f"Immediate lookup (within TTL):  {result}")

time.sleep(0.06)  # wait past the TTL
result = cache_ttl.get([1.0, 0.0, 0.0])
print(f"After TTL expired (60ms later): {result}")

# Add a fresh entry to demonstrate evict_expired()
cache_ttl.set([0.5, 0.5, 0.0], "Another answer")
evicted = cache_ttl.evict_expired()
print(f"\nAfter evict_expired(): removed {evicted} expired entries")
print(f"Remaining entries: {len(cache_ttl._entries)}")
print()
# Thread-safety note:
# time.time() reads are atomic in CPython. For concurrent writes, add threading.Lock().
print("Thread-safety: safe for single-threaded / async workflows (e.g. LangGraph).")

### Cache Warming

A cold cache starts with 0% hit rate. Three strategies to pre-warm before launch:

1. **Replay logs** — embed your top-100 historical queries and seed the cache before launch
2. **FAQ seeding** — for known-question systems (support chatbot, docs assistant), pre-embed all FAQs
3. **Synthetic generation** — use an LLM to generate paraphrase variants of common questions

Even seeding 50–100 entries dramatically improves day-one hit rates for narrow-domain applications.

**Implementation pattern:**
```python
def warm_cache(cache: SemanticCache, faq_pairs: list[tuple[str, str]], embed_fn) -> int:
    """Pre-warm cache from (question, answer) pairs. Returns entries added."""
    for question, answer in faq_pairs:
        vec = embed_fn(question)
        cache.set(vec, answer)
    return len(faq_pairs)
```

In [ ]:
# Part A — Cache warming simulation with mock embeddings
# Shows how a pre-warmed cache improves hit rate vs a cold cache.

FAQ_PAIRS = [
    ([1.0, 0.0, 0.0, 0.0], "Machine learning is a subset of AI that learns from data."),
    ([0.0, 1.0, 0.0, 0.0], "Deep learning uses multi-layer neural networks."),
    ([0.0, 0.0, 1.0, 0.0], "Python is a high-level, interpreted programming language."),
    ([0.0, 0.0, 0.0, 1.0], "A neural network is a system of interconnected nodes."),
    ([0.7, 0.7, 0.0, 0.0], "AI stands for Artificial Intelligence."),
]

# User queries that should hit the warm cache
LIVE_QUERIES = [
    ([0.99, 0.05, 0.0, 0.0], "What is machine learning?"),       # near FAQ[0]
    ([0.0, 0.99, 0.01, 0.0], "Explain deep learning"),           # near FAQ[1]
    ([0.0, 0.0, 0.99, 0.01], "Tell me about Python"),            # near FAQ[2]
    ([0.5, 0.0, 0.0, 0.5],   "What is an AI model?"),            # partial match
    ([0.3, 0.3, 0.3, 0.3],   "How does machine intelligence work?"),  # weak match
]

# Cold cache
cold_cache = SemanticCache(threshold=0.92)
cold_hits = sum(1 for vec, _ in LIVE_QUERIES if cold_cache.get(vec) is not None)

# Warm cache
warm_cache = SemanticCache(threshold=0.92)
for vec, answer in FAQ_PAIRS:
    warm_cache.set(vec, answer)

warm_hits = 0
for vec, query in LIVE_QUERIES:
    result = warm_cache.get(vec)
    if result:
        warm_hits += 1

n = len(LIVE_QUERIES)
print(f"Cold cache hit rate: {cold_hits}/{n} = {cold_hits/n:.0%}")
print(f"Warm cache hit rate: {warm_hits}/{n} = {warm_hits/n:.0%}")
print()
print(f"Cache warming added {len(FAQ_PAIRS)} entries and improved hit rate by {(warm_hits-cold_hits)/n:.0%}.")
print("In production: warm from top-N historical queries before serving live traffic.")

In [ ]:
# Part A — Production stats helper
# Computes summary metrics from a query log. Useful for threshold tuning.

def cache_stats_from_log(log: list[dict]) -> dict:
    """Compute summary stats from a list of {'hit': bool, 'latency_ms': int} log entries."""
    total = len(log)
    if total == 0:
        return {}
    hits = sum(1 for e in log if e.get("hit"))
    llm_latencies = [e["latency_ms"] for e in log if not e.get("hit") and "latency_ms" in e]
    return {
        "total_queries":       total,
        "cache_hits":          hits,
        "cache_misses":        total - hits,
        "hit_rate":            round(hits / total, 3),
        "llm_calls_saved":     hits,
        "avg_llm_latency_ms":  round(sum(llm_latencies) / len(llm_latencies)) if llm_latencies else 0,
        "estimated_savings_usd": round(hits * 0.001, 4),  # assume $0.001 per avoided LLM call
    }


# Simulate a realistic production log: ~35% hit rate after warm-up
import random
random.seed(42)

mock_log = [
    {
        "hit": random.random() < 0.35,
        "latency_ms": random.randint(200, 1200) if random.random() > 0.35 else 0,
    }
    for _ in range(100)
]

stats = cache_stats_from_log(mock_log)
print("Production stats (100 simulated queries, ~35% hit rate):\n")
for k, v in stats.items():
    print(f"  {k:<28}: {v}")

print()
print(f"At this hit rate, you save ~${stats['estimated_savings_usd']:.4f} per 100 queries.")
print("Scale to 10,000 queries/day → $" + f"{stats['estimated_savings_usd'] * 100:.2f}/day in avoided LLM costs.")

---
### Part A Recap

Before moving to live API calls, here is what the CPU-safe cells demonstrated:

| Cell | What it showed |
|---|---|
| cosine_sim demo | Identical → 1.0, orthogonal → 0.0, paraphrase → ~0.997 |
| SemanticCache walkthrough | Internal `_entries` list grows with each insertion |
| Threshold sweep | 0.85 = permissive, 0.92 = balanced, 0.99 = strict |
| Trace table | 2/5 queries hit cache at 0.92 (2 paraphrases caught) |
| Precision analysis | Below 0.90 false positives appear |
| LRU eviction | `OrderedDict` bounds cache size, evicts least-used entries |
| Exercise 1 | Class-level counters aggregate hits across instances |
| Exercise 2 | TTL expiry prevents stale answers |
| Cache warming | Pre-seeding 5 FAQs raised hit rate from 0% to 60% |
| Stats helper | Quantifies LLM calls saved and estimated cost reduction |

**Part B** repeats the 5-query demo with real `text-embedding-3-small` vectors
and live `gpt-5.4-nano` responses to confirm the mock behavior matches production.

---
## Part B — Live OpenAI API Calls

**Requires:** `OPENAI_API_KEY` in your `.env` file or shell environment.

Part B uses **real embeddings** from `text-embedding-3-small` and real LLM responses
from `gpt-5.4-nano`. The semantic cache is wired into the LangGraph workflow from
`src/workflow.py`.

**Cost estimate:** 5 queries × ~50 tokens each = ~250 tokens.
At $0.02/MTok that is **< $0.000005** — effectively free.

LLM calls (gpt-5.4-nano) run only on cache misses. This live corpus uses a calibrated
0.70 threshold: its measured ML and deep-learning paraphrase similarities are lower than the synthetic Part A vectors.

In [ ]:
# Part B — Setup: fail-fast API key check
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set.\n"
        "Add it to your .env file: OPENAI_API_KEY=sk-...\n"
        "Or export it: export OPENAI_API_KEY=sk-..."
    )
print("OPENAI_API_KEY is configured.")

In [ ]:
# Part B — Real embeddings: show dimensionality and first 5 values
import sys
sys.path.insert(0, ".")

from openai import OpenAI
from src.tools import embed

client = OpenAI(api_key=OPENAI_API_KEY)

QUERIES = [
    "What is machine learning?",
    "Can you explain machine learning to me?",  # near-duplicate → expect HIT
    "What is deep learning?",
    "How does deep learning work?",             # near-duplicate → expect HIT
    "What is a relational database?",           # new topic → expect MISS
]

print("Embedding 5 queries with text-embedding-3-small...\n")
embeddings = []
for q in QUERIES:
    vec = embed(q, client)
    embeddings.append(vec)
    preview = [round(x, 4) for x in vec[:5]]
    print(f"  '{q[:45]}'")
    print(f"    dims={len(vec)}  first_5={preview}")

print(f"\nEmbedding dimensionality: {len(embeddings[0])} (text-embedding-3-small)")

# Show real cosine similarities between query pairs
print("\nReal cosine similarities:")
pairs = [
    (0, 1, "ML vs ML paraphrase"),
    (2, 3, "DL vs DL paraphrase"),
    (0, 4, "ML vs DB (unrelated)"),
]
for i, j, label in pairs:
    sim = cosine_sim(embeddings[i], embeddings[j])
    will_hit = "HIT" if sim >= 0.70 else "MISS"
    print(f"  {label:<30}: {sim:.4f}  → {will_hit} at calibrated threshold 0.70")

In [ ]:
# Part B — Run the LangGraph semantic caching workflow
import json
from src.workflow import create_workflow

workflow = create_workflow()
config = {"configurable": {"client": client}}

print("Running semantic cache workflow with 5 queries...\n")
result = workflow.invoke({
    "queries": QUERIES,
    "threshold": 0.70,
    "responses": [],
    "cache_stats": {},
}, config=config)

print(f"{'#':<2}  {'Source':<12}  {'Query':<44}  Response snippet")
print("-" * 100)

llm_calls = 0
for i, r in enumerate(result["responses"], 1):
    if r["source"] == "llm":
        llm_calls += 1
        label = f"LLM {r['latency_ms']}ms"
    else:
        label = "CACHE HIT"
    snippet = r["response"][:48] + "..." if len(r["response"]) > 48 else r["response"]
    print(f"{i:<2}  {label:<12}  {r['query']:<44}  {snippet}")

stats = result["cache_stats"]
print()
print(f"Hit rate:          {stats['hit_rate'] * 100:.0f}%  ({stats['hits']} hits / {stats['hits'] + stats['misses']} queries)")
print(f"LLM calls made:    {llm_calls} of {len(QUERIES)}")
print(f"LLM calls saved:   {len(QUERIES) - llm_calls}")
print(f"Full cache stats:  {json.dumps(stats, indent=2)}")

---
## What's Next

- **GPTCache paper:** https://arxiv.org/abs/2310.01421 — the original semantic cache research with benchmarks
- **Example 141 (prompt-caching):** Anthropic-native caching with `cache_control` — caches at the system-prompt level, not the query level
- **Example 143 (context-compression):** Shrink input tokens before sending to the LLM — complementary to caching
- **Vector databases for production caches:** Pinecone, Weaviate, pgvector — persistent, horizontally scalable
- **Approximate nearest-neighbor search:** FAISS, HNSW — replace O(N) linear scan with O(log N) for caches with thousands of entries
- **LangChain semantic cache:** `from langchain.cache import InMemorySemanticCache` — drop-in with the same principles
- **Cache warming:** Pre-embed your top-100 historical queries before launch for a non-zero day-one hit rate